## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks demonstrates how to access the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: [terraclimate- overview](https://planetarycomputer.microsoft.com/dataset/terraclimate#overview) 

## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to "restart" the kernal so the Python libraries are available in the environment. This is done by selecting the "Connected" menu above the notebook (next to "Run all") and selecting the "restart kernal" link. Subsequent runs of the notebook do not require this "restart" process.

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

## Extracting TerraClimate Data Using API Calls

The API-based method allows us to efficiently access **TerraClimate** data for specific regions and time periods through the [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/), ensuring scalability and reproducibility of the process.

### Loading and Mapping TerraClimate Data
This section demonstrates how TerraClimate climate variables are loaded and mapped to sampling locations.

In [ ]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )
    return ds

In [ ]:
def filterg(ds, var):
    da = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))
    da = da.where(
        (da["lat"] > -35.18) & (da["lat"] < -21.72) &
        (da["lon"] > 14.97)  & (da["lon"] < 32.79),
        drop=True
    )
    df = da.to_dataframe().reset_index()
    df["time"] = df["time"].astype(str)
    df = df.rename(columns={"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"})
    print(f"Filtering for {var} completed, rows={len(df)}")
    return df

In [ ]:
def assign_nearest_climate(sa_df, climate_df, var_name):
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)
    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]
    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_values = []
    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']
        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]
        if subset.empty:
            climate_values.append(np.nan)
            continue
        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])
    return pd.DataFrame({var_name: climate_values})

### Extracting Data for Training Dataset

In [ ]:
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
display(Water_Quality_df.head(5))

In [ ]:
ds = load_terraclimate_dataset()
tc_vars = [
    "aet", "def", "pdsi", "pet", "ppt",
    "q", "soil", "srad", "swe",
    "tmax", "tmin",
    "vap", "vpd", "ws",
    "ppt_station_influence", "tmax_station_influence", "tmin_station_influence", "vap_station_influence"
]
available = set(ds.data_vars)
tc_vars_exist = [v for v in tc_vars if v in available]
print("Will extract:", tc_vars_exist)

Terraclimate_training_df = Water_Quality_df[["Latitude", "Longitude", "Sample Date"]].copy()
for i, v in enumerate(tc_vars_exist):
    if i % 3 == 0:
        ds = load_terraclimate_dataset()
    tc_parameter = filterg(ds, v)
    Terraclimate_training_df[v] = assign_nearest_climate(Water_Quality_df, tc_parameter, v)[v]

### Extracting Data for Validation Dataset

In [ ]:
Validation_df=pd.read_csv('submission_template.csv')
display(Validation_df.head())

In [ ]:
Terraclimate_validation_df = Validation_df[["Latitude", "Longitude", "Sample Date"]].copy()
for v in tc_vars_exist:
    tc_parameter = filterg(ds, v)
    Terraclimate_validation_df[v] = assign_nearest_climate(Validation_df, tc_parameter, v)[v]

Terraclimate_validation_df = Terraclimate_validation_df[["Latitude", "Longitude", "Sample Date"] + tc_vars_exist]

### Feature Engineering (Basic, Interactions & Anomalies)
套用領域知識建立氣候衍生特徵，提升模型預測水質的準確度。

In [ ]:
def add_basic_derived(df):
    df_new = df.copy()
    if 'tmax' in df.columns and 'tmin' in df.columns:
        df_new['temp_range'] = df_new['tmax'] - df_new['tmin']
        df_new['temp_mean'] = (df_new['tmax'] + df_new['tmin']) / 2
    if 'ppt' in df.columns and 'pet' in df.columns:
        df_new['net_precipitation'] = df_new['ppt'] - df_new['pet']
    return df_new

def add_interactions(df):
    df_new = df.copy()
    if 'q' in df.columns and 'ppt' in df.columns:
        df_new['runoff_ratio'] = df_new['q'] / (df_new['ppt'] + 0.001)
    if 'pdsi' in df.columns and 'soil' in df.columns:
        df_new['drought_soil_interaction'] = df_new['pdsi'] * df_new['soil']
    if 'srad' in df.columns and 'vpd' in df.columns:
        df_new['drying_power'] = df_new['srad'] * df_new['vpd']
    return df_new

def add_anomalies(df, ds, vars_for_anom):
    df_new = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(df_new['Sample Date']):
        df_new['Sample Date'] = pd.to_datetime(df_new['Sample Date'], dayfirst=True, errors='coerce')
    df_new['month'] = df_new['Sample Date'].dt.month
    for var in vars_for_anom:
        if var in df_new.columns:
            monthly_mean = df_new.groupby('month')[var].transform('mean')
            df_new[f'{var}_anomaly'] = df_new[var] - monthly_mean
    df_new = df_new.drop(columns=['month'])
    return df_new

In [ ]:
print("=== 對 Training 資料進行 1+2+4 特徵工程 ===")
Terraclimate_training_df = add_basic_derived(Terraclimate_training_df)
Terraclimate_training_df = add_interactions(Terraclimate_training_df)
vars_for_anom = ['ppt','pet','aet','def','pdsi','q','soil','tmax','tmin','vpd','srad']
Terraclimate_training_df = add_anomalies(Terraclimate_training_df, ds, vars_for_anom)

print("\n=== 對 Validation 資料進行相同處理 ===")
Terraclimate_validation_df = add_basic_derived(Terraclimate_validation_df)
Terraclimate_validation_df = add_interactions(Terraclimate_validation_df)
Terraclimate_validation_df = add_anomalies(Terraclimate_validation_df, ds, vars_for_anom)

print(f"\n🎉 完成！ Training shape: {Terraclimate_training_df.shape} | Validation shape: {Terraclimate_validation_df.shape}")

### Save Engineered Datasets

In [ ]:
Terraclimate_training_df.to_csv("terraclimate_features_training.csv", index=False)
Terraclimate_training_df.to_csv("/tmp/terraclimate_features_training_Gemini_v2.csv", index = False)

session.sql("""
    PUT file:///tmp/terraclimate_features_training_Gemini_v2.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print("Training Data Saved!")

In [ ]:
Terraclimate_validation_df.to_csv("terraclimate_features_validation_v2.csv", index=False)
Terraclimate_validation_df.to_csv("/tmp/terraclimate_features_validation_Gemini_v2.csv", index = False)

session.sql("""
    PUT file:///tmp/terraclimate_features_validation_Gemini_v2.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print("Validation Data Saved!")